# Supervised Fine-Tuning of Qwen2.5-1.5B using QLoRA

## Objective
This notebook performs Supervised Fine-Tuning (SFT) on **Qwen2.5-1.5B-Instruct** using **QLoRA** (Quantized Low-Rank Adaptation). QLoRA loads the base model in 4-bit precision and trains only lightweight LoRA adapter weights — reducing GPU memory from ~6GB to ~1.5GB, making it feasible on Kaggle's T4 GPU.

## Dataset
- **Source:** `sft_train.jsonl` (Intel/orca_dpo_pairs — 5,000 samples)
- **Format:** Each row has `prompt` and `response` fields
- **Split:** 95% train / 5% eval

## Workflow
1. Install and configure dependencies
2. Verify library versions
3. Define model + training hyperparameters
4. Inspect dataset file paths
5. Load and format dataset using Qwen chat template
6. Initialize tokenizer + token length analysis
7. Load model with 4-bit quantization
8. Attach LoRA adapters
9. Configure SFTTrainer and launch training
10. Save fine-tuned model

## 1. Environment Setup

Installing all required libraries with **pinned versions** to avoid dependency conflicts on Kaggle's T4 GPU.

> **Why `--no-deps`?**
> Kaggle pre-installs torch/torchvision. Using `--no-deps` upgrades only the target library without accidentally pulling an incompatible torch version.

| Library | Version | Purpose |
|---|---|---|
| `bitsandbytes` | 0.43.1 | 4-bit quantization (QLoRA) |
| `transformers` | 4.44.2 | Model + tokenizer loading |
| `peft` | 0.12.0 | LoRA adapter injection |
| `trl` | 0.11.4 | SFTTrainer |
| `accelerate` | 0.34.2 | Multi-GPU + mixed precision |

In [1]:
!pip install -q --force-reinstall --no-deps bitsandbytes==0.43.1

In [2]:
!pip install -q --force-reinstall --no-deps transformers==4.44.2

In [3]:
!pip install -q --force-reinstall --no-deps peft==0.12.0

In [4]:
!pip install -q --force-reinstall --no-deps trl==0.11.4

In [5]:
!pip install -q --force-reinstall --no-deps accelerate==0.34.2

In [6]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes

Found existing installation: bitsandbytes 0.43.1
Uninstalling bitsandbytes-0.43.1:
  Successfully uninstalled bitsandbytes-0.43.1
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [7]:
!pip install -q 'accelerate>=1.1.0'

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.11.4 requires tyro>=0.5.11, which is not installed.


In [8]:
!pip install -q --force-reinstall transformers==4.44.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.11.4 requires tyro>=0.5.11, which is not installed.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.3

## 2. Dependency Verification

Confirm all libraries are installed at the correct versions before loading any model.
If any version looks wrong, re-run the install cells above before continuing.

| Library | Expected Version |
|---|---|
| transformers | 4.44.2 |
| trl | 0.11.4 |
| peft | 0.12.0 |
| bitsandbytes | >= 0.43.1 |
| accelerate | >= 1.1.0 |

In [9]:
import transformers, trl, peft, bitsandbytes, accelerate

print(f"transformers  : {transformers.__version__}")
print(f"trl           : {trl.__version__}")
print(f"peft          : {peft.__version__}")
print(f"bitsandbytes  : {bitsandbytes.__version__}")
print(f"accelerate    : {accelerate.__version__}")

transformers  : 4.44.2
trl           : 0.11.4
peft          : 0.12.0
bitsandbytes  : 0.49.2
accelerate    : 1.13.0


In [10]:
!pip install -q liger-kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 10.2 MB/s eta 0:00:0000:01


In [11]:
!pip uninstall -y liger-kernel

Found existing installation: liger_kernel 0.8.0
Uninstalling liger_kernel-0.8.0:
  Successfully uninstalled liger_kernel-0.8.0


In [12]:
!pip install -q liger-kernel==0.3.1 transformers==4.44.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.0 MB/s eta 0:00:00


## 3. Model and Training Configuration

All hyperparameters are defined as constants here — easy to tune without hunting through the code.

### Model Settings
| Parameter | Value | Meaning |
|---|---|---|
| `MODEL_NAME` | Qwen2.5-1.5B-Instruct | Base model from HuggingFace |
| `MAX_LENGTH` | 512 | Max tokens per sample (longer sequences get truncated) |
| `OUTPUT_DIR` | `/kaggle/working/sft_model` | Where fine-tuned adapter weights are saved |

### LoRA Hyperparameters
| Parameter | Value | Meaning |
|---|---|---|
| `LORA_R` | 16 | Adapter rank — higher = more expressive, more memory |
| `LORA_ALPHA` | 32 | Scaling factor — effective multiplier = alpha/r = 2 |
| `LORA_DROPOUT` | 0.05 | Dropout on adapter layers for regularization |
| `LORA_TARGET` | q,k,v,o,gate,up,down | All attention + MLP projections targeted |

### Training Hyperparameters
| Parameter | Value | Meaning |
|---|---|---|
| `NUM_EPOCHS` | 3 | Full passes over dataset |
| `BATCH_SIZE` | 2 | Per-device batch size (T4 limit) |
| `GRAD_ACCUM` | 8 | Gradient accumulation steps |
| `LR` | 2e-4 | Learning rate |
| `LR_SCHEDULER` | cosine | Smooth LR decay for stable convergence |
| `WARMUP_RATIO` | 0.05 | 5% of steps used for LR warmup |

> **Effective batch size = BATCH_SIZE × GRAD_ACCUM = 2 × 8 = 16**

In [13]:
import torch
import json
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

2026-06-11 02:20:29.730530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781144429.908911     231 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781144429.963397     231 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781144430.392299     231 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781144430.392356     231 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781144430.392359     231 computation_placer.cc:177] computation placer alr

Torch: 2.10.0+cu128
CUDA available: True
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


In [14]:
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR  = "/kaggle/working/sft_model"
DATA_PATH   = "/kaggle/input/your-dataset-folder/sft_train.jsonl"  # ← update this path to your Kaggle input path
MAX_LENGTH  = 512

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET     = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ── Training ─────────────────────────────────────────────────────────────────
NUM_EPOCHS      = 3
BATCH_SIZE      = 2
GRAD_ACCUM      = 8
LR              = 2e-4
LR_SCHEDULER    = "cosine"
WARMUP_RATIO    = 0.05
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 0.3

# ── Eval ─────────────────────────────────────────────────────────────────────
EVAL_SPLIT      = 0.05
EVAL_STEPS      = 50
SAVE_STEPS      = 100
LOGGING_STEPS   = 10

print("Config set")
print(f"  LoRA rank: {LORA_R}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Target modules: {LORA_TARGET}")

Config set
  LoRA rank: 16
  Epochs: 3
  Effective batch size: 16
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 4. Dataset Inspection

Walk the Kaggle `/kaggle/input` directory to find the exact path of `sft_train.jsonl`.


In [15]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/sanidhyasrivastav67/sft-train/sft_train.jsonl


## 5. Load Raw Dataset

Load `sft_train.jsonl` into a DataFrame for a quick shape and sample check.
Expected: **5000 rows**, columns — `prompt`, `response`.

In [16]:
import json
import pandas as pd

def load_jsonl(file):
    with open(file) as f:
        return [json.loads(line) for line in f]

sft = pd.DataFrame(load_jsonl("/kaggle/input/datasets/sanidhyasrivastav67/sft-train/sft_train.jsonl"))
print(f"SFT shape: {sft.shape}")
print(f"Columns: {sft.columns.tolist()}")
print("\nSFT sample:")
display(sft.head())

SFT shape: (5000, 2)
Columns: ['prompt', 'response']

SFT sample:


,prompt,response
0,Your skies will be blue and filled with stars ...,"Your skies will be blue, and filled with stars..."
1,Instructions: The task is to generate text bas...,Task Explanation: The task is asking to genera...
2,"In this task, you will be presented with a pre...",This task is asking you to analyze the relatio...
3,Answer by taking a quote from the following ar...,The quote provided does not mention the specif...
4,Definition: Given a sequence of actions to nav...,"turn right, jump, turn right, jump, run twice"


## 6. Format Dataset using Qwen Chat Template

Each `prompt`/`response` pair is structured as a user-assistant conversation and passed through `apply_chat_template`. This ensures the model sees the **exact same special tokens** during SFT as it was pre-trained on — critical for instruction-following quality.

Final formatted text looks like:
```
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
```

In [17]:
def load_and_format_dataset(data_path, tokenizer, eval_split=0.05):
    raw = []
    with open(data_path) as f:
        for line in f:
            line = line.strip()
            if line:
                raw.append(json.loads(line))

    formatted = []
    for item in raw:
        # Build chat messages from your prompt/response fields
        messages = [
            {"role": "user",      "content": item["prompt"]},
            {"role": "assistant", "content": item["response"]}
        ]
        # Apply Qwen chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        formatted.append({"text": text})

    dataset = Dataset.from_list(formatted)

    # Train / eval split
    split = dataset.train_test_split(test_size=eval_split, seed=42)
    print(f"Train samples : {len(split['train'])}")
    print(f"Eval  samples : {len(split['test'])}")
    print(f"\nSample formatted text (truncated):\n{split['train'][0]['text'][:300]}")
    return split["train"], split["test"]

In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Qwen2.5 uses eos as pad — keep consistent
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"   # required for SFTTrainer with causal LM

print(f"Vocab size    : {tokenizer.vocab_size}")
print(f"Pad token     : {tokenizer.pad_token}")
print(f"EOS token     : {tokenizer.eos_token}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size    : 151643
Pad token     : <|endoftext|>
EOS token     : <|im_end|>


In [19]:
train_dataset, eval_dataset = load_and_format_dataset("/kaggle/input/datasets/sanidhyasrivastav67/sft-train/sft_train.jsonl", tokenizer, EVAL_SPLIT)

Train samples : 4750
Eval  samples : 250

Sample formatted text (truncated):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I am trying to decide whether it's worth it to invest in this film proposal. Can you help me answer a few questions? If you can't, please say "No I can't".  Question: Who paid Tyler to 


## 7. Tokenizer Initialization

Load the Qwen2.5 tokenizer.

**Key decisions:**
- `pad_token = eos_token` — Qwen2.5 has no dedicated pad token; this is the standard fix for decoder-only models
- `padding_side = "right"` — Required by SFTTrainer; left-padding causes label misalignment in causal LM training

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.padding_side  = "right"  

print(f"Vocab size:    {tokenizer.vocab_size}")
print(f"Pad token:     {tokenizer.pad_token}")
print(f"Padding side:  {tokenizer.padding_side}")

# quick token length check on the dataset
lengths = [len(tokenizer(s["text"], truncation=False)["input_ids"]) for s in train_dataset.select(range(200))]
print(f"\nToken length stats (first 200 samples):")
print(f"  min: {min(lengths)}")
print(f"  max: {max(lengths)}")
print(f"  avg: {sum(lengths)//len(lengths)}")
print(f"  over MAX_LENGTH ({MAX_LENGTH}): {sum(1 for l in lengths if l > MAX_LENGTH)}")

Vocab size:    151643
Pad token:     <|im_end|>
Padding side:  right

Token length stats (first 200 samples):
  min: 57
  max: 1485
  avg: 389
  over MAX_LENGTH (512): 55


## 8. Load Model with 4-bit Quantization (QLoRA)

The model is loaded using `BitsAndBytesConfig` with **NF4 quantization**, reducing VRAM from ~6GB (fp16) to ~1.5GB.

| Config | Value | Reason |
|---|---|---|
| `bnb_4bit_quant_type = "nf4"` | NormalFloat4 | Better than int4 for normally distributed weights |
| `bnb_4bit_use_double_quant = True` | Double quant | Quantizes the quantization constants too — extra memory saving |
| `bnb_4bit_compute_dtype = bfloat16` | bf16 | Faster compute during forward/backward pass |

In [21]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = "nf4",
    bnb_4bit_compute_dtype     = torch.bfloat16,   
    bnb_4bit_use_double_quant  = True,             
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
model.config.use_cache = False                    
model.config.pretraining_tp = 1

# count total vs trainable params
total  = sum(p.numel() for p in model.parameters())
print(f"Model loaded")
print(f"Total params: {total/1e6:.1f}M")

Loading model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded
Total params: 888.6M


## 9. Prepare Model for QLoRA Training

`prepare_model_for_kbit_training` does two things:
1. Enables **gradient checkpointing** — saves VRAM by recomputing activations during backward pass
2. Casts **LayerNorm layers to fp32** — prevents instability when training a 4-bit quantized model

In [22]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
print("Model prepared for kbit training")

Model prepared for kbit training


## 10. Attach LoRA Adapters

LoRA injects small trainable rank-decomposition matrices into the attention and MLP layers. Only these adapter weights (~1-2% of total params) are updated — the base model stays completely frozen.

The `print_trainable_parameters()` call below confirms exactly how many params are being trained.

In [23]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 11. Training Configuration (SFTConfig)

Key decisions explained:

| Setting | Value | Why |
|---|---|---|
| `gradient_checkpointing` | True | Saves VRAM — essential on T4 |
| `paged_adamw_8bit` | optimizer | Memory-efficient optimizer from bitsandbytes |
| `bf16 = True` | precision | Faster on modern GPUs; switch to `fp16` if NaN issues appear |
| `load_best_model_at_end` | True | Auto-restores checkpoint with lowest eval_loss after training |
| `packing = False` | off | Keeps samples separate — avoids cross-sample attention leakage |

In [24]:
sft_config = SFTConfig(
    # output
    output_dir                  = OUTPUT_DIR,

    # data
    max_seq_length              = MAX_LENGTH,
    dataset_text_field          = "text",         
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,            

    # optimizer
    learning_rate               = LR,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    optim                       = "paged_adamw_8bit",  
    # precision
    bf16                        = True,
    fp16                        = False,

    # eval & saving
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,              
    load_best_model_at_end      = True,           
    metric_for_best_model       = "eval_loss",

    # logging
    logging_steps               = LOGGING_STEPS,
    report_to                   = "none",

    # packing — off
    packing                     = False,
)

print("SFTConfig set")

SFTConfig set


## 12. Custom Loss Logger Callback

A lightweight callback that prints `train_loss` and `eval_loss` at every eval step directly to stdout — since we're not using W&B or TensorBoard on Kaggle.

Losses are also stored in lists (`train_losses`, `eval_losses`) for post-training analysis if needed.

In [25]:
# simple callback to print train + eval loss at each eval step
# so you can see if model is converging without wandb
class LossLogger(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses  = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        if "loss" in logs:
            self.train_losses.append((step, logs["loss"]))
            print(f"  Step {step:>4} | train_loss: {logs['loss']:.4f}", end="")
        if "eval_loss" in logs:
            self.eval_losses.append((step, logs["eval_loss"]))
            print(f"  |  eval_loss: {logs['eval_loss']:.4f}", end="")
        print()

loss_logger = LossLogger()
print("Callback ready")

Callback ready


## 13. Initialize Trainer and Start Fine-Tuning

`SFTTrainer` handles the full training loop:
- Tokenization and sequence packing
- Gradient accumulation
- Checkpoint saving at `save_steps`
- Evaluation at `eval_steps`
- Restoring best checkpoint at end

**Expected behavior:**
- `eval_loss` should decrease steadily across epochs
- Training complete in ~45-60 mins on Kaggle T4

In [ ]:
trainer = SFTTrainer(
    model           = model,
    args            = sft_config,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    tokenizer       = tokenizer,
    callbacks       = [loss_logger],
)

print("Starting SFT training...")
print(f"  Total steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto'}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}\n")

trainer.train()

print("\nTraining complete")

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Starting SFT training...
  Total steps: auto
  Epochs: 3
  Effective batch size: 16



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss,Validation Loss
50,1.422800,1.482902
100,1.385400,1.458034
150,1.460400,1.445483


  Step   10 | train_loss: 1.9247
  Step   20 | train_loss: 1.7305
  Step   30 | train_loss: 1.4782
  Step   40 | train_loss: 1.4538
  Step   50 | train_loss: 1.4228
  |  eval_loss: 1.4829
  Step   60 | train_loss: 1.3882
  Step   70 | train_loss: 1.4037
  Step   80 | train_loss: 1.3969
  Step   90 | train_loss: 1.4470
  Step  100 | train_loss: 1.3854
  |  eval_loss: 1.4580


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  Step  110 | train_loss: 1.4141
  Step  120 | train_loss: 1.3083
  Step  130 | train_loss: 1.3654
  Step  140 | train_loss: 1.4147
  Step  150 | train_loss: 1.4604
  |  eval_loss: 1.4455
  Step  160 | train_loss: 1.4256
  Step  170 | train_loss: 1.4328
